# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path
import time
from itertools import product
import math

from rich.pretty import pprint

import numpy as np
import cv2
import pandas as pd


from enderscope.scan_patterns import snake
from enderscope.serial import list_ports, Stage
from enderscope.enderlights import Enderlights

import panel as pn

from enderleaf.image import lap_var, to_pil, safe_pil_resize, canny, find_circles
from enderleaf.tools import time_method
from enderleaf.preview_panel import preview
from enderleaf.qr_reader import get_qr_data, get_points_extremes
from enderleaf.tools import ensure_folder, format_datetime
from enderscope.enderlights_pi import Enderlights

## Setup

## Constants

In [ ]:
TEMPLATE_X_SIZE = 200
TEMPLATE_Y_SIZE = 200
TEMPLATE_SIZE = (TEMPLATE_X_SIZE, TEMPLATE_Y_SIZE)
ROW_COUNT, COL_COUNT = 9, 9
LEAF_DIAM = 17
CAM_RES = (4608, 2592)
CROP_TOP = 400
CROP_BOTTOM = 350
CROP_LEFT = 1200
CROP_RIGHT = 1200
COLUMNS = [i + 1 for i in range(COL_COUNT)]
ROWS = [chr(65 + i) for i in range(ROW_COUNT)]
DST_FLD = Path(".").joinpath("output")

## Functions

In [ ]:
def acquire_image(stage: Stage, lights: Enderlights, pos=None):
    if pos is not None:
        stage.move_position(pos)
    stage.finish_moves()
    lights.shutter(True)
    image = preview().capture_array()
    lights.shutter(False)
    return image

In [ ]:
def get_qr_pos(stage, lights):
    qr_data = get_qr_data(acquire_image(stage=stage, lights=lights))
    if qr_data["retval"] is False:
        raise ValueError("Unable to detect QR code")
    min_x, min_y, max_x, max_y = get_points_extremes(points=qr_data["points"][0])
    return (min_x + max_x) // 2, (min_y + max_y) // 2


In [ ]:
def get_best_z(
    stage,
    lights,
    pos,
    min_rel_z: float = 5.0,
    max_rel_z: float = 5.0,
):
    zrange = np.array(range(min_rel_z, max_rel_z, 1))
    mxScore = -1
    bestZ = 0
    lights.shutter(True)

    for z in zrange:
        stage.move_position([pos[0], pos[1], z + pos[2]])
        stage.finish_moves()
        img = preview().capture_array()
        grayImage = (
            np.float32(img[:, :, 0])
            + np.float32(img[:, :, 1])
            + np.float32(img[:, :, 2])
        ) / 3
        score = lap_var(grayImage)
        if score > mxScore:
            mxScore = score
            bestZ = z
    lights.shutter(False)

    stage.move_position([pos[0], pos[1], pos[2]])
    stage.finish_moves()

    return bestZ + pos[2]

In [ ]:
def center_qr_code(stage, lights, start_height):
    preview().set_crop(top=0, bottom=0, left=0, right=0)
    stage.move_absolute(
        30,
        30,
        get_best_z(
            stage=stage,
            lights=lights,
            pos=[30, 30, start_height],
            min_rel_z=-10,
            max_rel_z=10,
        ),
    )
    cx, cy = CAM_RES[0] // 2, CAM_RES[1] // 2
    qr_cx, qr_cy = get_qr_pos(stage, lights)
    step_x, step_y = -10 if cx > qr_cx else 10, 10 if cy > qr_cy else -10

    stage.move_relative(step_x, step_y)
    new_qr_cx, new_qr_cy = get_qr_pos(stage, lights)

    stage.move_relative(-step_x, -step_y)
    stage.move_relative(
        abs(cx - qr_cx) * step_x / abs(new_qr_cx - qr_cx),
        abs(cy - qr_cy) * step_y / abs(new_qr_cy - qr_cy),
    )
    return stage.get_position()

In [ ]:
@time_method
def run_job(stage, lights, speed, start_height):
    preview().on_request_focus_close()
    cx, cy, z = preview().center_on_qr_code()
    # cx, cy, z = center_qr_code(stage, lights, start_height=start_height)
    qr_data = get_qr_data(acquire_image(stage=stage, lights=lights))
    if qr_data["retval"] is False:
        raise ValueError("Unable to detect QR code")
    exp_name = qr_data["info"][0]
    min_x, min_y, max_x, max_y = get_points_extremes(points=qr_data["points"][0])

    positions = snake(cols=COL_COUNT, rows=ROW_COUNT) * [
        # steps
        TEMPLATE_X_SIZE / ROW_COUNT,
        TEMPLATE_Y_SIZE / COL_COUNT,
    ] + [
        # origin
        cx,
        cy,
    ]

    # preview().set_crop(
    #     top=min_y, bottom=CAM_RES[1] - max_y, left=min_x, right=CAM_RES[0] - max_x
    # )
    # z = get_best_z(
    #     stage=stage,
    #     lights=lights,
    #     pos=[cx, cy, start_height],
    #     min_rel_z=-10,
    #     max_rel_z=10,
    # )

    preview().set_crop(
        top=CROP_TOP, bottom=CROP_BOTTOM, left=CROP_LEFT, right=CROP_RIGHT
    )

    image_data = []
    try:
        exp, inoc, _ = exp_name.split("_")
        ensure_folder(DST_FLD.joinpath(exp, inoc))
    except:
        exp, inoc = "test", "i1"
        ensure_folder(DST_FLD.joinpath(exp, inoc))

    # for p in [positions[0], positions[8], positions[80], positions[72]]:
    #     image = acquire_image(stage=stage, lights=lights, pos=np.append(p, z))

    for p, (c, r) in zip(positions, list(product(COLUMNS, ROWS))):
        image = acquire_image(stage=stage, lights=lights, pos=np.append(p, z))
        file_name = f"{exp_name}#{r}#{c}#{format_datetime()}"
        cv2.imwrite(
            str(DST_FLD.joinpath(exp, inoc).joinpath(file_name).with_suffix(".png")),
            cv2.cvtColor(image, cv2.COLOR_RGB2BGR),
        )
        image_data.append(
            {
                "exp_name": exp_name,
                "row": r,
                "col": c,
                "image": image,
                "position": p,
                "file_name": file_name,
            }
        )

    # stage.move_position((30, TEMPLATE_Y_SIZE, 100))
    stage.finish_moves()
    return image_data

## Initialize hardware

In [ ]:
# list available serial ports
# stage_port = None
# ports = list_ports()
# for port in ports:
#     if "USB Serial" in port.description:
#         stage_port = port
#         print("* " + str(port))
#     else:
#         print("  " + str(port))

In [ ]:
# stage = Stage(stage_port, 115200)

In [ ]:
# lights = Enderlights()
# lights.shutter(True)
# time.sleep(1)
# lights.shutter(False)

In [ ]:
# success = stage.safe_home()
# if success:
#     stage.move_position((30, TEMPLATE_Y_SIZE, 100))
#     stage.finish_moves()
# else:
#     raise ConnectionError("Unable to home")

## Initialize Preview

In [ ]:
preview().show()

In [ ]:
preview().rest()

## Run Acquisitions

In [ ]:
images = run_job(stage=preview()._stage, lights=lights, speed=6000, start_height=36)

## Visualize Results

In [ ]:
sel_image = pn.widgets.IntSlider(
    name="Select image",
    start=0,
    end=len(images) - 1,
    value=0,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()
json_data = pn.pane.JSON()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    ph_image.object = safe_pil_resize(to_pil(images[index]["image"]), 600,600)
    json_data.object = {k:str(v) for k, v in images[index].items() if k != "image"}


on_index_changed(sel_image.value)

pn.Column(ph_image, json_data, sel_image)

## Grab circles

In [ ]:
factor = 4
idx = 3

In [ ]:
image = images[idx]["image"]
image = cv2.resize(image, (image.shape[1] // factor, image.shape[0] // factor))
edges = canny(image=image, color_space="rgb", channel="blue")
to_pil(edges)

In [ ]:
accums, cx, cy, radii = find_circles(
    edges=edges,
    radii=np.arange(450 // factor, 550 // factor, 20 // factor),
    max_circles=3,
)
pprint(accums, cx, cy, radii)
for cy, cx, radius in zip(cy, cx, radii):
    print(radius * factor)
    image = cv2.circle(
        image, (cx, cy), radius, (255, 0, 255), 10
    )

to_pil(image)